In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import pwd, os
USERNAME=pwd.getpwuid(os.getuid()).pw_name
DIANNE = f"/home/{USERNAME}/dianne-codebase"
sys.path.insert(0, f"{DIANNE}/dianne-core")
sys.path.insert(0, f"{DIANNE}/dianne-utils")
sys.path.insert(0, f"{DIANNE}/dianne-viewer")
import dianne_core
import dianne_utils as dianne
import dianne_viewer as viewer
dianne.setNotebookWidth(100)
import pandas as pd
import matplotlib.colors as mcolors
import scanpy as sc
def createH2(slide, mpath=None):
    df_temp = pd.read_csv(f'{mpath}/{slide}.matrix-H.csv', header=None)
    df_temp.iloc[:2,:2] *= 0.25 / 0.2208187960959237
    sname = f'{mpath}/{slide}.matrix-H-2.csv'
    if not os.path.isfile(sname):
        df_temp.to_csv(sname, index=False, header=False)
    return
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)
idm = './identity-matrix.csv'
pd.DataFrame([[1,0,0], [0,1,0], [0,0,1]]).to_csv(idm, index=False, header=False)

In [ ]:
stqPath = '/projects/activities/kappsen-tmc/USERS/domans/Fetal-membranes-analysis/STQ-results/'
xePath = '/projects/activities/kappsen-tmc/USERS/domans/dev-MIQ-sequential/run-xenium-ranger'
mpath = '/projects/activities/kappsen-tmc/USERS/domans/Fetal-membranes-analysis/SenReg-results'
sotPath = '/projects/activities/kappsen-tmc/USERS/domans/dev-MIQ-sequential/spatial-omics-tools/miq/mesmer/results'

samples = ['MC_PLACM_0011_5k', 'MC_PLACM_0003_5k']
F = 2
fname = f'features/false-{F}-ctranspath_features.tsv.gz'
patch_size = 8
ts, mpp, tile_size = dianne.loadSTQParams(stqPath + samples[0], F)
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne_core.loadDataAndPreparePatches(samples, stqPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
sizes = {s: ads[s].shape[0] for s in samples}
print(f'Prepared {patchesCDFs.shape[0]} patches')

runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5, multiplier=2, erode=True)
savefn = dianne.makeSaveFn(patchCoordinates, ads, samples, qs, ts, mpp, PCMA_alpha=0.8, tile_size=tile_size, patch_size=patch_size, body_overlap=0.25, classifierPaths=classifierPaths)
loadfn = dianne.makeLoadFn(classifierPaths)
listfn = dianne.makeListFn(classifierPaths)

xenium_bundle_paths = {'MC_PLACM_0011_5k': f'{xePath}/region1-PLACM-11-reseg-v7',
                       'MC_PLACM_0003_5k': f'{xePath}/region2-PLACM-3-reseg-v7'}

for sample in samples:
    createH2(sample + '.he', mpath)
xenium_to_he_matrices = {sample: f'{mpath}/{sample}.he.matrix-H-2.csv' for sample in samples}

mif_images = {'MC_PLACM_0011_5k': f'{sotPath}/XE5k-PLACM-0011-v7/adjusted-all-channels.ome.tif',
              'MC_PLACM_0003_5k': f'{sotPath}/XE5k-PLACM-0003-v7/adjusted-all-channels.ome.tif'}
identity_matrices = {sample: './identity-matrix.csv' for sample in samples}

drawings = viewer.create_viewer(samples, mif_images, height="800px", run_inference_fn=runfn, sample_sizes=sizes,
                                xenium_mpp=0.2125, max_cells=15000, matrices=identity_matrices, xenium_bundle_paths=xenium_bundle_paths,
                                secondary_images=imgs, secondary_matrices=xenium_to_he_matrices, draw_on_secondary=True,
                                save_func=savefn, load_func=loadfn, list_names_func=listfn)[1]